# Spike de exploración — `contract_check` (T-68)

> Artefacto del **paso 4** del flujo (`notebook_writer`). Prototipa y **visualiza** cada historia de
> `610_features/contract_check/definition.md` (HU-01…HU-11) antes de escribir la spec.
> **Es un spike, no producto:** al pasar a `app/src/`, la spec y el bucle TDD **reescriben** la lógica; no se
> copia-pega desde aquí. El notebook queda como documentación de referencia.

## Qué se prototipa
Un subcomando nuevo — **`zlk contract check <CLIENTE>`** — que resuelve
`<clients_root>/<CLIENTE>/input/contract_data.yaml` e invoca el motor **ya existente** `load_contract`
(`app/src/zeroleak/config/contract.py`), mostrando su respuesta. **El motor no se modifica**: hoy existe,
está verificado y habla trece mensajes congelados (M-01…M-13), pero **nadie lo invoca**. Esta feature no
construye motor: **conecta el motor que ya existe con la persona que lo necesita**.

## Reglas que este spike respeta
- 🔒 **C-01 (Datos en Bóveda):** tenants y YAMLs **exclusivamente sintéticos**, en un directorio temporal que
  se borra al final. Ningún dato real de cliente entra aquí.
- 🚫 **No se toca producción:** el motor y `cli.py` se **importan**; el prototipo de fachada vive solo en el
  notebook. La Celda 14 verifica por `sha256` que `contract.py` quedó intacto.
- 🧪 **Evidencia, no promesas:** los errores que se demuestran están **capturados a propósito**; las
  afirmaciones de ausencia ("no lee CSV", "no escribe") llevan **guarda de no-vacuidad** (L-17).

## Mapa de celdas
| Celda | Contenido | Traza |
|---|---|---|
| 1 | Núcleo vigente importado + `sha256` del motor | HU-10 |
| 2 | Datos sintéticos: 5 tenants + fixtures YAML | HU-11 |
| 3 | **Prototipo** de la fachada `contract check` | HU-01…HU-08 |
| 4 | `clients_root` con la convención vigente | HU-06 |
| 5 | Camino feliz → exit 0 + señal de éxito | HU-01 |
| 6 | Contrato recién scaffoldeado → M-01 exacto | HU-04 |
| 7 | Inválido por esquema → mensaje **verbatim** | HU-02 |
| 8 | YAML roto → `ContractParseError`, distinguible | HU-03 |
| 9 | `FileNotFoundError` del motor → traducción de dominio | HU-05 |
| 10 | 🔍 **Punto abierto:** exit codes vs prefijo | (gate) |
| 11 | Invocaciones mal formadas → uso | HU-07 |
| 12 | No escribe nada (foto `sha256` + `mtime`) | HU-08 |
| 13 | No lee ningún CSV (2 espías + guarda, L-17) | HU-09 |
| 14 | Motor intacto, C-01, resumen de trazas, limpieza | HU-10, HU-11 |

## ⚠️ Estado de ejecución
El agente que escribió este notebook **no dispone de entorno de ejecución** en su contexto, por lo que las
celdas están **sin salidas**. Antes del gate del paso 5 debe ejecutarse de arriba a abajo (el proyecto exige
Python 3.13):

```powershell
py -3.13 -m jupyter nbconvert --to notebook --execute --inplace 610_features/contract_check/contract_check.ipynb
```

**Sin salidas visibles el notebook no sirve para el gate.** Las celdas llevan `assert` en cada afirmación
importante: si la ejecución llega al final sin fallar, cada HU quedó demostrada; si alguna revienta, la
evidencia es igual de valiosa (mostraría que el enfoque prototipado no se sostiene y debe corregirse **antes**
de escribir la spec).

## Celda 1 — El núcleo **vigente**, importado (no reescrito)

Se importa el motor tal como está exportado hoy (`load_contract`, `Contract`, las dos excepciones y las
constantes congeladas M-01/M-07/M-12) y la fachada vigente (`cli`), para reutilizar su `_clients_root()` y su
patrón de despacho. Se registra el `sha256` de `config/contract.py` **al abrir**, para compararlo al cerrar
(HU-10).

In [1]:
# =============================================================================
# Celda 1 — Nucleo VIGENTE importado: se CONSUME, no se redefine ni se modifica
# Traza: HU-10 (el motor queda intacto) — soporte de todas las demas celdas
# =============================================================================
import builtins
import contextlib
import hashlib
import io
import os
import shutil
import sys
import tempfile
import textwrap
import traceback
from pathlib import Path

# Raiz del repo desde la ubicacion del notebook (610_features/contract_check/)
RAIZ_REPO = Path.cwd()
while not (RAIZ_REPO / "app" / "src").is_dir() and RAIZ_REPO != RAIZ_REPO.parent:
    RAIZ_REPO = RAIZ_REPO.parent
SRC = RAIZ_REPO / "app" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# --- El motor y la fachada VIGENTES: se importan, NO se copian ni se tocan ----
from zeroleak import cli                       # fachada vigente (client new / ingest)
from zeroleak.core.scaffold import create_client
import zeroleak.config.contract as contract_mod
from zeroleak.config.contract import (
    Contract,
    ContractParseError,
    ContractSchemaError,
    load_contract,
    _MENSAJE_RAIZ_INVALIDA,    # M-01 (raiz sin cuerpo)
    _MENSAJE_COLUMNAS_VACIA,   # M-07 (columnas vacias)
    _MENSAJE_YAML_INVALIDO,    # M-12 (YAML sintacticamente invalido)
)

CONTRACT_PY = Path(contract_mod.__file__)
SHA_CONTRACT_INICIAL = hashlib.sha256(CONTRACT_PY.read_bytes()).hexdigest()

print("Python                :", sys.version.split()[0])
print("raiz del repo         :", RAIZ_REPO)
print("motor importado desde :", CONTRACT_PY)
print("fachada vigente       :", Path(cli.__file__))
print("sha256(contract.py) al abrir el spike:", SHA_CONTRACT_INICIAL)
print("   (se compara al cerrar, Celda 14 -> HU-10: el core no se modifica)")
print()
print("_USAGE vigente hoy en cli.py:")
print(textwrap.indent(cli._USAGE, "    "))
print()
print("Subcomandos que despacha `main` hoy: client new, ingest  ->  falta `contract check`")
print("Excepciones del motor disponibles  :", ContractParseError.__name__, "|", ContractSchemaError.__name__)
print("Mensajes congelados que se usaran aqui:")
print("   M-01:", repr(_MENSAJE_RAIZ_INVALIDA))
print("   M-07:", repr(_MENSAJE_COLUMNAS_VACIA))
print("   M-12:", repr(_MENSAJE_YAML_INVALIDO))

Python                : 3.12.10
raiz del repo         : C:\Users\USUARIO\Documents\TripleS\ZeroLeak_Application
motor importado desde : C:\Users\USUARIO\Documents\TripleS\ZeroLeak_Application\app\src\zeroleak\config\contract.py
fachada vigente       : C:\Users\USUARIO\Documents\TripleS\ZeroLeak_Application\app\src\zeroleak\cli.py
sha256(contract.py) al abrir el spike: 564b44a3f046328cfcb8da977cd7c6b2de5caf0d8661f36e84f759f77754c118
   (se compara al cerrar, Celda 14 -> HU-10: el core no se modifica)

_USAGE vigente hoy en cli.py:
    uso: zlk client new <NOMBRE_CLIENTE>
         zlk ingest <CLIENTE> <ruta>...

Subcomandos que despacha `main` hoy: client new, ingest  ->  falta `contract check`
Excepciones del motor disponibles  : ContractParseError | ContractSchemaError
Mensajes congelados que se usaran aqui:
   M-01: "la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: {cuerpo!r})"
   M-07: 'la lista de columnas no puede estar vacía'
   M-12: 'el a

## Celda 2 — Datos sintéticos ("matrices de mentiras") en un directorio temporal

🔒 **C-01 (HU-11).** Todo lo que sigue es **inventado** y vive en un `tempdir` que se borra en la última
celda. Los tenants se crean con el `create_client` **vigente** (así el caso M-01 es el real, no una
imitación) y `ZEROLEAK_CLIENTS_ROOT` apunta al temporal, de modo que el spike **nunca** toca el `clients/`
del repo.

| tenant | `contract_data.yaml` | traza |
|---|---|---|
| `cliente_ok` | válido, 2 archivos | HU-01 |
| `cliente_scaffold` | **intacto** desde `client new` | HU-04 (M-01) |
| `cliente_esquema` | inválido por esquema | HU-02 |
| `cliente_roto` | sintaxis YAML rota | HU-03 |
| `cliente_sin_contrato` | borrado a propósito | HU-05 |
| `cliente_fantasma` | *no existe* | HU-05 |

Además se siembra un **CSV sintético en `data/bronze/`** (3 filas: `test1@correo.com`, …). Existe solo para
que la Celda 13 pueda demostrar que el comando **no lo abre** (frontera D-21): sin un CSV presente, esa
prueba sería vacua.

In [2]:
# =============================================================================
# Celda 2 — Datos sinteticos ("matrices de mentiras") en un directorio TEMPORAL
# Traza: HU-11 (C-01, Datos en Boveda). Nada real de ningun cliente.
# =============================================================================
TMP = Path(tempfile.mkdtemp(prefix="zlk_spike_contract_check_"))
CLIENTS_ROOT = TMP / "clients"
CLIENTS_ROOT.mkdir(parents=True)

# HU-06: el comando resolvera el tenant con la convencion vigente. Apuntandola
# al temporal, el spike NUNCA toca el `clients/` real del repo.
os.environ["ZEROLEAK_CLIENTS_ROOT"] = str(CLIENTS_ROOT)

# --- Fixture 1: contrato VALIDO multi-archivo (forma D-18) -------------------
YAML_VALIDO_MULTI = """\
# contract_data.yaml — SINTETICO (spike). Estructura, no datos.
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
        - {nombre: correo,     tipo: string,  nulable: false, llave: false}
        - {nombre: alta,       tipo: date,    nulable: true,  llave: false}
    - nombre: ventas.csv
      columnas:
        - {nombre: venta_id,   tipo: integer, nulable: false, llave: true}
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: false}
        - {nombre: monto,      tipo: float,   nulable: false, llave: false}
"""

# --- Fixture 2: INVALIDO por esquema (columnas vacias en el 2do archivo, M-07)
YAML_ESQUEMA_INVALIDO = """\
# contract_data.yaml — SINTETICO, invalido por ESQUEMA (columnas vacias)
contract_data:
  archivos:
    - nombre: clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
    - nombre: ventas.csv
      columnas: []
"""

# --- Fixture 3: YAML sintacticamente ROTO (comilla sin cerrar) ---------------
YAML_ROTO = """\
# contract_data.yaml — SINTETICO, sintaxis YAML rota a proposito
contract_data:
  archivos:
    - nombre: "clientes.csv
      columnas:
        - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
"""

# --- Tenants sinteticos, creados con el `client new` VIGENTE ------------------
TENANTS = {}
for nombre in ("cliente_ok", "cliente_scaffold", "cliente_esquema",
               "cliente_roto", "cliente_sin_contrato"):
    TENANTS[nombre] = create_client(nombre, CLIENTS_ROOT)


def _escribir_contrato(cliente, texto):
    ruta = TENANTS[cliente] / "input" / "contract_data.yaml"
    ruta.write_text(texto, encoding="utf-8")
    return ruta


_escribir_contrato("cliente_ok", YAML_VALIDO_MULTI)          # HU-01
_escribir_contrato("cliente_esquema", YAML_ESQUEMA_INVALIDO)  # HU-02
_escribir_contrato("cliente_roto", YAML_ROTO)                 # HU-03
# cliente_scaffold: se deja TAL CUAL lo dejo `client new`     (HU-04)
(TENANTS["cliente_sin_contrato"] / "input" / "contract_data.yaml").unlink()  # HU-05
# cliente_fantasma: NO se crea (HU-05, tenant inexistente)

# --- CSV sintetico ya "ingerido" en bronze -----------------------------------
# Existe SOLO para poder demostrar en la Celda 13 que `contract check` NO lo abre
# (frontera D-21). 3 filas de mentiras: nombres y correos falsos.
CSV_BRONZE = TENANTS["cliente_ok"] / "data" / "bronze" / "clientes.csv"
CSV_BRONZE.write_text(
    "cliente_id,correo,alta\n"
    "1,test1@correo.com,2026-01-01\n"
    "2,test2@correo.com,2026-01-02\n"
    "3,test3@correo.com,2026-01-03\n",
    encoding="utf-8",
)

print("clients_root SINTETICO (temporal):", CLIENTS_ROOT)
print("ZEROLEAK_CLIENTS_ROOT            :", os.environ["ZEROLEAK_CLIENTS_ROOT"])
print()
print(f"{'tenant':<22}{'contract_data.yaml':<22}proposito")
print("-" * 78)
_PROPOSITO = {
    "cliente_ok": ("valido, 2 archivos", "HU-01 camino feliz"),
    "cliente_scaffold": ("scaffold intacto", "HU-04 -> M-01"),
    "cliente_esquema": ("invalido (esquema)", "HU-02 -> ContractSchemaError"),
    "cliente_roto": ("sintaxis rota", "HU-03 -> ContractParseError"),
    "cliente_sin_contrato": ("BORRADO", "HU-05 tenant sin contrato"),
}
for nombre, (estado, proposito) in _PROPOSITO.items():
    print(f"{nombre:<22}{estado:<22}{proposito}")
print(f"{'cliente_fantasma':<22}{'(no existe)':<22}HU-05 tenant inexistente")

print("\ncontract_data.yaml de cliente_scaffold, tal como lo deja `zlk client new`:")
print(textwrap.indent(
    (TENANTS["cliente_scaffold"] / "input" / "contract_data.yaml").read_text(encoding="utf-8"),
    "    | "))
print("CSV sintetico en bronze (para la frontera D-21 de la Celda 13):", CSV_BRONZE)

clients_root SINTETICO (temporal): C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients
ZEROLEAK_CLIENTS_ROOT            : C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients

tenant                contract_data.yaml    proposito
------------------------------------------------------------------------------
cliente_ok            valido, 2 archivos    HU-01 camino feliz
cliente_scaffold      scaffold intacto      HU-04 -> M-01
cliente_esquema       invalido (esquema)    HU-02 -> ContractSchemaError
cliente_roto          sintaxis rota         HU-03 -> ContractParseError
cliente_sin_contrato  BORRADO               HU-05 tenant sin contrato
cliente_fantasma      (no existe)           HU-05 tenant inexistente

contract_data.yaml de cliente_scaffold, tal como lo deja `zlk client new`:
    | # contract_data.yaml — Contrato de Datos (completar).
    | contract_data:

CSV sintetico en bronze (para la frontera D-21 de la Celda 13): C:\Users\USUA

## Celda 3 — El prototipo: `zlk contract check <CLIENTE>` como **fachada delgada**

Prototipo de los ~15 renglones que la feature agregaría a `cli.py`. Imita el patrón vigente de
`_dispatch_ingest`: **resolver ruta → comprobar precondiciones → invocar el motor → traducir a exit code**.
No contiene lógica de validación: la única llamada que valida es `load_contract(contrato)`.

> ⚠️ **Es un spike, no producción.** Al construir la feature, la spec y el bucle TDD **reescriben** esta
> lógica en `app/src/`; no se copia-pega desde aquí.

El parámetro `variante` (`"A"` / `"B"`) **existe solo en el spike** para poder contrastar en la Celda 10 las
dos formas de distinguir `parse` de `schema`. El código de producción tendrá **una sola**, la que el humano
elija en el gate.

In [3]:
# =============================================================================
# Celda 3 — PROTOTIPO de la fachada `zlk contract check <CLIENTE>`
# Traza: HU-01..HU-08. SPIKE, NO PRODUCCION: el bucle TDD reescribira esto
# desde la spec; aqui solo se explora la forma.
# =============================================================================

# _USAGE propuesto: el vigente + UNA linea (HU-07)
_USAGE_SPIKE = (
    "uso: zlk client new <NOMBRE_CLIENTE>\n"
    "     zlk ingest <CLIENTE> <ruta>...\n"
    "     zlk contract check <CLIENTE>"
)

EXIT_OK = 0
EXIT_PRECONDICION = 2      # patron `ingest`: TenantNotFoundError -> 2
EXIT_PARSE_A = 3           # variante A: exit codes distintos (Celda 10)
EXIT_ESQUEMA_A = 4
EXIT_ERROR_B = 1           # variante B: mismo exit code, prefijo distinto
PREFIJO_PARSE_B = "ERROR DE SINTAXIS YAML: "
PREFIJO_ESQUEMA_B = "ERROR DE ESQUEMA: "


def _parse_contract_check_args(argv):
    """`contract check <CLIENTE>` -> cliente; None si no es este subcomando.
    len == 3 EXACTO, como `_parse_client_new_name` (HU-07)."""
    if len(argv) == 3 and argv[0] == "contract" and argv[1] == "check":
        return argv[2]
    return None


def _dispatch_contract_check(client, variante="A"):
    """Fachada delgada: resolver -> precondiciones -> load_contract -> traducir.
    Ni un gramo de logica de validacion propia (HU-10, CA-12 de `ingest`)."""
    clients_root = cli._clients_root()                      # HU-06: convencion vigente
    tenant_dir = clients_root / client
    contrato = tenant_dir / "input" / "contract_data.yaml"

    # HU-05: precondiciones ANTES de invocar al motor. El motor asume que el
    # YAML existe (D-22) y hace open() sin capturar FileNotFoundError.
    if not tenant_dir.is_dir():
        print(f"Tenant inexistente o incompleto: {client!r}", file=sys.stderr)
        return EXIT_PRECONDICION
    if not contrato.is_file():
        print(f"El tenant {client!r} no tiene contrato: falta {contrato}", file=sys.stderr)
        return EXIT_PRECONDICION

    try:
        contract = load_contract(contrato)                  # el MOTOR, tal cual
    except ContractParseError as exc:                       # HU-03
        if variante == "A":
            print(str(exc), file=sys.stderr)
            return EXIT_PARSE_A
        print(f"{PREFIJO_PARSE_B}{exc}", file=sys.stderr)
        return EXIT_ERROR_B
    except ContractSchemaError as exc:                      # HU-02 / HU-04
        if variante == "A":
            print(str(exc), file=sys.stderr)                # VERBATIM, sin adornos
            return EXIT_ESQUEMA_A
        print(f"{PREFIJO_ESQUEMA_B}{exc}", file=sys.stderr)
        return EXIT_ERROR_B

    # HU-01: senal de exito que identifica QUE se valido
    nombres = ", ".join(a.nombre for a in contract.archivos)
    print(f"OK  contrato valido: {client} — {contrato}")
    print(f"    {len(contract.archivos)} archivo(s) declarado(s): {nombres}")
    return EXIT_OK


def spike_main(argv, variante="A"):
    """`main(argv)` con el subcomando nuevo. El despacho vigente se REUSA tal
    cual (`cli._parse_*` / `cli._dispatch_*`): el spike no lo reescribe."""
    client = _parse_contract_check_args(argv)
    if client is not None:
        return _dispatch_contract_check(client, variante=variante)

    name = cli._parse_client_new_name(argv)
    if name is not None:
        return cli._dispatch_client_new(name)

    ingest_args = cli._parse_ingest_args(argv)
    if ingest_args is not None:
        return cli._dispatch_ingest(*ingest_args)

    print(_USAGE_SPIKE, file=sys.stderr)                    # HU-07
    return 1


# --- Utilidades del spike para ver exit code / stdout / stderr por separado ---
def correr(argv, variante="A"):
    out, err = io.StringIO(), io.StringIO()
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err):
        code = spike_main(list(argv), variante=variante)
    return code, out.getvalue(), err.getvalue()


def mostrar(titulo, argv, variante="A"):
    code, out, err = correr(argv, variante=variante)
    print("=" * 78)
    print(f"$ zlk {' '.join(argv)}".ljust(52) + f"[variante {variante}]")
    print(f"  {titulo}")
    print("-" * 78)
    print(f"  exit code : {code}")
    print(f"  stdout    : {out.strip() if out.strip() else '(vacio)'}")
    print(f"  stderr    : {err.strip() if err.strip() else '(vacio)'}")
    print("=" * 78)
    return code, out, err


print("Prototipo de fachada listo. Despacho propuesto:")
print(textwrap.indent(_USAGE_SPIKE, "    "))
print("\nExit codes del prototipo (variante A):")
print(f"    exito                  -> {EXIT_OK}")
print(f"    precondicion (HU-05)   -> {EXIT_PRECONDICION}   (mismo patron que `ingest`)")
print(f"    ContractParseError     -> {EXIT_PARSE_A}")
print(f"    ContractSchemaError    -> {EXIT_ESQUEMA_A}")
print(f"    invocacion mal formada -> 1")
print("\nEl motor se INVOCA, no se reimplementa: la unica llamada de validacion es")
print("`load_contract(contrato)`.")

Prototipo de fachada listo. Despacho propuesto:
    uso: zlk client new <NOMBRE_CLIENTE>
         zlk ingest <CLIENTE> <ruta>...
         zlk contract check <CLIENTE>

Exit codes del prototipo (variante A):
    exito                  -> 0
    precondicion (HU-05)   -> 2   (mismo patron que `ingest`)
    ContractParseError     -> 3
    ContractSchemaError    -> 4
    invocacion mal formada -> 1

El motor se INVOCA, no se reimplementa: la unica llamada de validacion es
`load_contract(contrato)`.


## Celda 4 — HU-06: `clients_root` con la **convención vigente** (`ZEROLEAK_CLIENTS_ROOT` → `./clients`)

La fachada no inventa convención: invoca `cli._clients_root()`, exactamente la misma función que ya usan
`client new` e `ingest`. Se prueba la variable definida, el **fallback** a `./clients`, y —evidencia
end-to-end— que **mover el root cambia lo que el comando ve**.

In [4]:
# =============================================================================
# Celda 4 — HU-06: la resolucion de clients_root es la CONVENCION VIGENTE
# =============================================================================
print("La fachada NO inventa convencion: llama a `cli._clients_root()`, la misma")
print("funcion que ya usan `client new` e `ingest`.\n")

print("ZEROLEAK_CLIENTS_ROOT =", os.environ.get("ZEROLEAK_CLIENTS_ROOT"))
print("cli._clients_root()   =", cli._clients_root())
assert cli._clients_root() == CLIENTS_ROOT
print("   -> con la variable definida, gana la variable:", cli._clients_root() == CLIENTS_ROOT)

# Fallback: sin la variable de entorno
guardado = os.environ.pop("ZEROLEAK_CLIENTS_ROOT")
print("\n(variable de entorno retirada temporalmente)")
print("cli._clients_root()   =", repr(cli._clients_root()))
assert cli._clients_root() == Path("clients")
print("   -> sin la variable, cae a ./clients:", cli._clients_root() == Path("clients"))
os.environ["ZEROLEAK_CLIENTS_ROOT"] = guardado
print("\n(variable restaurada) cli._clients_root() =", cli._clients_root())

# Y la ruta del contrato se deriva por la misma convencion de tenant
ruta = cli._clients_root() / "cliente_ok" / "input" / "contract_data.yaml"
print("\nruta del contrato resuelta para 'cliente_ok':")
print("   ", ruta)
print("    existe:", ruta.is_file())

# Evidencia end-to-end: mover el root cambia lo que el comando ve
otro_root = TMP / "otro_clients_root"
otro_root.mkdir()
os.environ["ZEROLEAK_CLIENTS_ROOT"] = str(otro_root)
code, out, err = correr(["contract", "check", "cliente_ok"])
print("\napuntando ZEROLEAK_CLIENTS_ROOT a un root VACIO:", otro_root)
print("   exit:", code, "| stderr:", err.strip())
print("   -> el comando obedece la variable de entorno, no una ruta cableada")
os.environ["ZEROLEAK_CLIENTS_ROOT"] = str(CLIENTS_ROOT)
code, out, err = correr(["contract", "check", "cliente_ok"])
print("\nrestaurado el root sintetico -> exit:", code, "| stdout:", out.strip().splitlines()[0])
assert code == 0
print("\nHU-06 OK: misma convencion, cero aprendizaje nuevo para el humano.")

La fachada NO inventa convencion: llama a `cli._clients_root()`, la misma
funcion que ya usan `client new` e `ingest`.

ZEROLEAK_CLIENTS_ROOT = C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients
cli._clients_root()   = C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients
   -> con la variable definida, gana la variable: True

(variable de entorno retirada temporalmente)
cli._clients_root()   = WindowsPath('clients')
   -> sin la variable, cae a ./clients: True

(variable restaurada) cli._clients_root() = C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients

ruta del contrato resuelta para 'cliente_ok':
    C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients\cliente_ok\input\contract_data.yaml
    existe: True

apuntando ZEROLEAK_CLIENTS_ROOT a un root VACIO: C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\otro_clients_root
   exit: 2 | stderr: Tenant inexistente 

## Celda 5 — HU-01: camino feliz · contrato válido multi-archivo → **exit 0** + señal de éxito

Primero se llama al motor **directo** (para ver qué devuelve realmente: un `Contract` tipado con sus dos
archivos y sus columnas), y después el **mismo** contrato a través del comando. La HU pide que el humano
sepa que el motor leyó **su** contrato: por eso la señal propuesta incluye **el tenant, la ruta exacta y los
archivos declarados**. La forma exacta del mensaje es un **punto abierto** que fija la spec.

In [5]:
# =============================================================================
# Celda 5 — HU-01: camino feliz (contrato valido multi-archivo)
# =============================================================================
contrato_ok = TENANTS["cliente_ok"] / "input" / "contract_data.yaml"

# (1) El motor, llamado directo: que es exactamente lo que la fachada va a mostrar
contrato = load_contract(contrato_ok)
print("load_contract devolvio:", type(contrato).__name__, "| es Contract:", isinstance(contrato, Contract))
print("archivos declarados   :", [a.nombre for a in contrato.archivos])
print()
for archivo in contrato.archivos:
    print(f"  {archivo.nombre}  ({len(archivo.columnas)} columnas)")
    for col in archivo.columnas:
        print(f"     - {col.nombre:<12} tipo={col.tipo.value:<9} nulable={str(col.nulable):<6} llave={col.llave}")

# (2) El mismo contrato, a traves del COMANDO
code, out, err = mostrar("HU-01 — contrato valido multi-archivo", ["contract", "check", "cliente_ok"])

print("\n--- Verificaciones ---")
print("exit code 0                                  :", code == EXIT_OK)
print("la senal de exito va por STDOUT              :", out.strip() != "")
print("stderr vacio                                 :", err.strip() == "")
print("la senal IDENTIFICA al tenant                :", "cliente_ok" in out)
print("la senal IDENTIFICA el contrato que se leyo  :", str(contrato_ok) in out)
print("la senal dice QUE se valido (los archivos)   :", "clientes.csv" in out and "ventas.csv" in out)
assert code == EXIT_OK and err.strip() == "" and "cliente_ok" in out

print("\nPARA LA SPEC (punto abierto declarado en definition.md): la forma exacta del")
print("mensaje de exito. El spike propone: 'OK  contrato valido: <cliente> — <ruta>' +")
print("recuento y nombres de los archivos declarados. El razonamiento: la HU pide que")
print("el humano sepa que el motor leyo SU contrato y no otro -> la RUTA es la que")
print("aporta esa certeza; los nombres de archivo dejan ver de un vistazo si el motor")
print("entendio lo que el humano creia haber escrito.")

load_contract devolvio: Contract | es Contract: True
archivos declarados   : ['clientes.csv', 'ventas.csv']

  clientes.csv  (3 columnas)
     - cliente_id   tipo=integer   nulable=False  llave=True
     - correo       tipo=string    nulable=False  llave=False
     - alta         tipo=date      nulable=True   llave=False
  ventas.csv  (3 columnas)
     - venta_id     tipo=integer   nulable=False  llave=True
     - cliente_id   tipo=integer   nulable=False  llave=False
     - monto        tipo=float     nulable=False  llave=False
$ zlk contract check cliente_ok                     [variante A]
  HU-01 — contrato valido multi-archivo
------------------------------------------------------------------------------
  exit code : 0
  stdout    : OK  contrato valido: cliente_ok — C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients\cliente_ok\input\contract_data.yaml
    2 archivo(s) declarado(s): clientes.csv, ventas.csv
  stderr    : (vacio)

--- Verificaciones ---
e

## Celda 6 — HU-04: el contrato **recién scaffoldeado** (sin cuerpo) → M-01, camino de primera clase

`cliente_scaffold` se deja **tal como lo dejó `client new`**: `contract_data:` con la raíz presente y sin
cuerpo. Es la **primera corrida realista** de cualquier humano recién creado su tenant. No debe ser un
traceback: debe ser la señal útil *"tu contrato está vacío, complétalo"* — **M-01**, verificado por
**igualdad exacta** contra la constante congelada `_MENSAJE_RAIZ_INVALIDA`.

In [6]:
# =============================================================================
# Celda 6 — HU-04: primera corrida real — el contrato recien scaffoldeado -> M-01
# =============================================================================
contrato_scaffold = TENANTS["cliente_scaffold"] / "input" / "contract_data.yaml"
print("Contenido EXACTO que `zlk client new` deja en el tenant (scaffold.py:_CONTRACT_DATA_YAML):")
print(textwrap.indent(contrato_scaffold.read_text(encoding="utf-8"), "    | "))
print("   -> `contract_data:` sin cuerpo; PyYAML lo parsea a None (no es un YAML roto)")

code, out, err = mostrar("HU-04 — tenant recien creado por `client new`",
                         ["contract", "check", "cliente_scaffold"])

esperado_m01 = _MENSAJE_RAIZ_INVALIDA.format(cuerpo=None)
print("\n--- Verificaciones ---")
print("M-01 esperado (desde la constante congelada):")
print("   ", repr(esperado_m01))
print("stderr del comando (verbatim):")
print("   ", repr(err.strip()))
print("igualdad EXACTA con M-01:", err.strip() == esperado_m01)
print("exit != 0                :", code != 0)
assert err.strip() == esperado_m01, "el mensaje no es M-01 exacto"
assert code != 0

print("\nHU-04 OK: el caso de PRIMERA CORRIDA no es un borde ni un traceback:")
print("es la senal util 'tu contrato esta vacio, completalo' (M-01, el fix de T-56).")
print("\nHALLAZGO para spec_writer: la fachada no necesita NINGUNA rama especial para")
print("este caso — el motor ya lo cubre. Es un CA de test, no de codigo.")

Contenido EXACTO que `zlk client new` deja en el tenant (scaffold.py:_CONTRACT_DATA_YAML):
    | # contract_data.yaml — Contrato de Datos (completar).
    | contract_data:

   -> `contract_data:` sin cuerpo; PyYAML lo parsea a None (no es un YAML roto)
$ zlk contract check cliente_scaffold               [variante A]
  HU-04 — tenant recien creado por `client new`
------------------------------------------------------------------------------
  exit code : 4
  stdout    : (vacio)
  stderr    : la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: None)

--- Verificaciones ---
M-01 esperado (desde la constante congelada):
    "la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: None)"
stderr del comando (verbatim):
    "la clave raíz 'contract_data' debe contener un mapa con la lista 'archivos' (se encontró: None)"
igualdad EXACTA con M-01: True
exit != 0                : True

HU-04 OK: el caso de PRIMERA CORRIDA n

## Celda 7 — HU-02: contrato **inválido por esquema** → el mensaje del motor **verbatim**

El fixture viola el esquema (`ventas.csv` con `columnas: []` → **M-07**). Se comparan **dos caminos** sobre
el mismo YAML:
1. `load_contract(...)` llamado **directo** (lo que vería un desarrollador) → `str(exc)`.
2. El **comando** → `stderr`.

La evidencia de "verbatim" es la **igualdad exacta entre ambos**, más la igualdad exacta contra la constante
congelada `_MENSAJE_COLUMNAS_VACIA` (M-07) con su localizador `archivo[i] 'nombre'` (D-25a).

In [7]:
# =============================================================================
# Celda 7 — HU-02: contrato invalido por esquema -> mensaje del motor VERBATIM
# =============================================================================
ruta_esq = TENANTS["cliente_esquema"] / "input" / "contract_data.yaml"
print("YAML sintetico invalido por esquema (ventas.csv con `columnas: []`):")
print(textwrap.indent(YAML_ESQUEMA_INVALIDO, "    "))

# (1) El motor, llamado DIRECTO (como lo llamaria un desarrollador)
try:
    load_contract(ruta_esq)
    tipo_motor, mensaje_motor = None, None
except ContractSchemaError as exc:
    tipo_motor, mensaje_motor = type(exc).__name__, str(exc)

print("excepcion del motor:", tipo_motor)
print("mensaje del motor  :", repr(mensaje_motor))

# (2) El mismo contrato, a traves del COMANDO
code, out, err = mostrar("HU-02 — contrato invalido por esquema", ["contract", "check", "cliente_esquema"])

print("\n--- Verificaciones ---")
print("exit != 0                                    :", code != 0)
print("stdout vacio (el error NO va por stdout)     :", out.strip() == "")
print("stderr == str(exc) del motor (VERBATIM)      :", err.strip() == mensaje_motor)

# Igualdad EXACTA contra la constante congelada M-07 reconstruida
esperado_m07 = f"archivo[1] 'ventas.csv': {_MENSAJE_COLUMNAS_VACIA}"
print("\nM-07 esperado (reconstruido desde la constante congelada + localizador D-25a):")
print("   ", repr(esperado_m07))
print("igualdad EXACTA del mensaje del motor con M-07:", mensaje_motor == esperado_m07)

assert code != 0
assert out.strip() == ""
assert err.strip() == mensaje_motor, "la fachada reescribio el mensaje del motor"
assert mensaje_motor == esperado_m07

print("\nHU-02 OK: la fachada no reescribe, no adorna, no traduce.")
print("\nNOTA DE METODO para spec_writer: la asercion `stderr == str(exc) del motor`")
print("prueba lo VERBATIM para CUALQUIER mensaje (incluidos M-06/M-08/M-09/M-10, que")
print("mezclan texto de Pydantic y no son reconstruibles a mano). La igualdad exacta")
print("contra la constante _MENSAJE_* aplica donde el texto es reconstruible (M-01,")
print("M-07...). La spec puede usar ambas: una por CA de mensaje concreto, otra como")
print("propiedad general de la fachada.")

YAML sintetico invalido por esquema (ventas.csv con `columnas: []`):
    # contract_data.yaml — SINTETICO, invalido por ESQUEMA (columnas vacias)
    contract_data:
      archivos:
        - nombre: clientes.csv
          columnas:
            - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}
        - nombre: ventas.csv
          columnas: []

excepcion del motor: ContractSchemaError
mensaje del motor  : "archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía"
$ zlk contract check cliente_esquema                [variante A]
  HU-02 — contrato invalido por esquema
------------------------------------------------------------------------------
  exit code : 4
  stdout    : (vacio)
  stderr    : archivo[1] 'ventas.csv': la lista de columnas no puede estar vacía

--- Verificaciones ---
exit != 0                                    : True
stdout vacio (el error NO va por stdout)     : True
stderr == str(exc) del motor (VERBATIM)      : True

M-07 esperado (recons

## Celda 8 — HU-03: YAML **sintácticamente roto** → `ContractParseError`, distinguible del de esquema

El YAML de `cliente_roto` tiene una comilla sin cerrar: PyYAML no puede ni construir el documento. El motor
lo captura antes de llegar al esquema y lo distingue como `ContractParseError` con M-12 (+ el detalle crudo
de PyYAML, que localiza línea y columna). La celda cierra comparando **lado a lado** el caso de sintaxis con
el de esquema: esa **distinguibilidad** es el corazón de la HU.

In [8]:
# =============================================================================
# Celda 8 — HU-03: YAML sintacticamente roto -> ContractParseError
# =============================================================================
print("El YAML sintetico roto (comilla sin cerrar):")
print(textwrap.indent(YAML_ROTO, "    "))

ruta_roto = TENANTS["cliente_roto"] / "input" / "contract_data.yaml"
try:
    load_contract(ruta_roto)
    excepcion_motor, mensaje_motor = None, None
except (ContractParseError, ContractSchemaError) as exc:
    excepcion_motor, mensaje_motor = type(exc).__name__, str(exc)

print("excepcion del motor:", excepcion_motor)
print("mensaje del motor  :", mensaje_motor)

code, out, err = mostrar("HU-03 — YAML sintacticamente roto", ["contract", "check", "cliente_roto"])

print("\n--- Verificaciones ---")
print("es ContractParseError, NO ContractSchemaError :", excepcion_motor == "ContractParseError")
print("stderr verbatim (== str(exc) del motor)      :", err.strip() == mensaje_motor)
print("empieza por M-12 congelado                   :", mensaje_motor.startswith(_MENSAJE_YAML_INVALIDO))
print("M-12 =", repr(_MENSAJE_YAML_INVALIDO))
print("...y despues antecede el detalle crudo de PyYAML (linea/columna del error)")
assert excepcion_motor == "ContractParseError"
assert err.strip() == mensaje_motor
assert code != 0

print("\n--- DISTINGUIBILIDAD (el corazon de HU-03) ---")
code_esq, _, err_esq = correr(["contract", "check", "cliente_esquema"])
print(f"cliente_roto    (sintaxis) -> exit {code}      | {err.strip().splitlines()[0][:52]}")
print(f"cliente_esquema (esquema)  -> exit {code_esq}      | {err_esq.strip().splitlines()[0][:52]}")
print("\nexit codes distintos entre sintaxis y esquema:", code != code_esq)
print("(esta es la VARIANTE A; la Celda 10 contrasta A vs B — decision del gate)")

El YAML sintetico roto (comilla sin cerrar):
    # contract_data.yaml — SINTETICO, sintaxis YAML rota a proposito
    contract_data:
      archivos:
        - nombre: "clientes.csv
          columnas:
            - {nombre: cliente_id, tipo: integer, nulable: false, llave: true}

excepcion del motor: ContractParseError
mensaje del motor  : el archivo YAML del contrato es sintácticamente inválido: while scanning a quoted scalar
  in "C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients\cliente_roto\input\contract_data.yaml", line 4, column 15
found unexpected end of stream
  in "C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients\cliente_roto\input\contract_data.yaml", line 7, column 1
$ zlk contract check cliente_roto                   [variante A]
  HU-03 — YAML sintacticamente roto
------------------------------------------------------------------------------
  exit code : 3
  stdout    : (vacio)
  stderr    : el archivo YAML del con

## Celda 9 — HU-05: el **hueco del motor** (`FileNotFoundError`) y su traducción

### 🔍 El caso feo (lo que el humano necesita ver)
`load_contract` hace `open(path)` **sin capturar `FileNotFoundError`** (`contract.py:213`). Sobre un tenant
inexistente —o uno sin `input/contract_data.yaml`— el motor **propaga un traceback crudo de infraestructura**.
Esta celda lo demuestra con evidencia real (excepción **capturada a propósito**, no un fallo del notebook) y
luego muestra cómo la fachada lo traduce.

**Lo importante: el core no se toca.** Por D-22 el motor *asume* que el YAML existe; es la **CLI** quien
comprueba la precondición **antes** de invocarlo — exactamente el patrón que `ingest` ya usa con
`TenantNotFoundError` → exit 2.

In [9]:
# =============================================================================
# Celda 9 — HU-05: el hueco REAL del motor y su traduccion por la fachada
# =============================================================================
print("#" * 78)
print("# (1) EVIDENCIA DEL HUECO: el motor, llamado directo, propaga infraestructura")
print("#" * 78)

HUECOS = [
    ("tenant inexistente", CLIENTS_ROOT / "cliente_fantasma" / "input" / "contract_data.yaml"),
    ("tenant sin contrato", TENANTS["cliente_sin_contrato"] / "input" / "contract_data.yaml"),
]
for etiqueta, ruta in HUECOS:
    print(f"\n--- {etiqueta}: load_contract({ruta.name}) ---")
    print("    la ruta existe?:", ruta.exists())
    try:
        load_contract(ruta)
        print("    (no fallo?!)")
    except ContractSchemaError as exc:
        print("    ContractSchemaError:", exc)
    except ContractParseError as exc:
        print("    ContractParseError:", exc)
    except FileNotFoundError as exc:
        print("    ", type(exc).__name__, "->", exc)
        print("     traceback crudo (ultimas 3 lineas) — esto es lo que veria el humano:")
        for linea in traceback.format_exc().strip().splitlines()[-3:]:
            print("       ", linea)

print("\n   => Confirmado: `load_contract` hace open(path) SIN capturar FileNotFoundError")
print("      (contract.py ~213). No es un bug del motor: por D-22 el motor asume que el")
print("      YAML existe. Cubrir esa precondicion es trabajo de la FACHADA.")

print("\n" + "#" * 78)
print("# (2) LA FACHADA lo traduce ANTES de invocar al motor (sin tocar el core)")
print("#" * 78)
code_f, out_f, err_f = mostrar("HU-05 — tenant inexistente", ["contract", "check", "cliente_fantasma"])
code_s, out_s, err_s = mostrar("HU-05 — tenant existente SIN input/contract_data.yaml",
                               ["contract", "check", "cliente_sin_contrato"])

for code, err in ((code_f, err_f), (code_s, err_s)):
    assert code == EXIT_PRECONDICION, "la precondicion debe tener su propio exit code"
    assert "Traceback" not in err and "FileNotFoundError" not in err, "se filtro infraestructura"
print("\nHU-05 OK: mensaje de dominio + exit", EXIT_PRECONDICION, "en ambos casos;")
print("ningun traceback ni FileNotFoundError llego a stderr.")
print("\nPARA LA SPEC: el spike usa DOS mensajes distintos (tenant ausente vs contrato")
print("ausente) con el MISMO exit code 2, imitando el patron de `ingest`")
print("(TenantNotFoundError -> 2). Queda abierto si la spec quiere dos exit codes.")

##############################################################################
# (1) EVIDENCIA DEL HUECO: el motor, llamado directo, propaga infraestructura
##############################################################################

--- tenant inexistente: load_contract(contract_data.yaml) ---
    la ruta existe?: False
     FileNotFoundError -> [Errno 2] No such file or directory: 'C:\\Users\\USUARIO\\AppData\\Local\\Temp\\zlk_spike_contract_check_xpie8cu9\\clients\\cliente_fantasma\\input\\contract_data.yaml'
     traceback crudo (ultimas 3 lineas) — esto es lo que veria el humano:
            with open(path, "r", encoding="utf-8") as fh:
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\USUARIO\\AppData\\Local\\Temp\\zlk_spike_contract_check_xpie8cu9\\clients\\cliente_fantasma\\input\\contract_data.yaml'

--- tenant sin contrato: load_contract(contract_data.yaml) ---
    la ruta existe?: False
     File

## Celda 10 — 🔍 Punto abierto para la spec: **cómo distinguir `parse` de `schema`** en la salida

El `feature_contract.md` deja esta decisión explícitamente a la spec: *"Cómo se distinguen —mismo exit code
con prefijo distinto, o exit codes distintos— lo fija la spec"*. Esta celda **no decide**: pone las dos
alternativas una al lado de la otra con salidas reales, para que el humano elija en el gate.

| | Variante **A** | Variante **B** |
|---|---|---|
| Mecanismo | exit codes distintos (parse=3, schema=4) | mismo exit (1) + prefijo en el texto |
| Texto del motor | intacto, verbatim | prefijado |

El criterio decisivo aparece al final de la celda: **¿sigue `stderr` siendo igual, carácter a carácter, al
mensaje que emite el motor?** — porque el CA-2 del contrato exige verificarlo *por igualdad exacta contra la
constante `_MENSAJE_*`, no por subcadena laxa*.

In [10]:
# =============================================================================
# Celda 10 — PUNTO ABIERTO: como distinguir parse de schema en la salida
# Se contrastan las dos opciones con salidas reales. NO se decide aqui.
# =============================================================================
CASOS_DISTINCION = [
    ("cliente_roto", "ContractParseError"),
    ("cliente_esquema", "ContractSchemaError"),
    ("cliente_scaffold", "ContractSchemaError (M-01)"),
    ("cliente_ok", "— (valido)"),
]

for variante, titulo in (("A", "VARIANTE A — exit codes distintos, texto del motor INTACTO"),
                         ("B", "VARIANTE B — mismo exit code (1), PREFIJO distinto")):
    print("=" * 78)
    print(titulo)
    print("=" * 78)
    print(f"{'cliente':<22}{'excepcion':<28}{'exit':<6}stderr (primera linea)")
    print("-" * 78)
    for cliente, excepcion in CASOS_DISTINCION:
        code, out, err = correr(["contract", "check", cliente], variante=variante)
        primera = (err.strip().splitlines() or ["(vacio)"])[0]
        print(f"{cliente:<22}{excepcion:<28}{code:<6}{primera[:44]}")
    print()

# --- El criterio que decide: ¿sigue siendo VERBATIM el mensaje del motor? -----
print("=" * 78)
print("LA TENSION QUE EL HUMANO DEBE RESOLVER EN EL GATE")
print("=" * 78)
for cliente in ("cliente_roto", "cliente_esquema"):
    ruta = TENANTS[cliente] / "input" / "contract_data.yaml"
    try:
        load_contract(ruta)
        del_motor = "(no fallo)"
    except (ContractParseError, ContractSchemaError) as exc:
        del_motor = str(exc)
    _, _, err_a = correr(["contract", "check", cliente], variante="A")
    _, _, err_b = correr(["contract", "check", cliente], variante="B")
    print(f"\n{cliente}:")
    print("   mensaje del motor          :", repr(del_motor[:60]))
    print("   variante A, stderr verbatim:", err_a.strip() == del_motor)
    print("   variante B, stderr verbatim:", err_b.strip() == del_motor)

print("""
LECTURA DEL SPIKE (para spec_writer; la decision es del humano):

  Variante A — exit codes distintos (parse=3, schema=4), texto sin tocar
    (+) stderr sigue siendo IGUAL, carácter a carácter, al mensaje del motor:
        cumple el CA-2 del feature_contract ("verbatim, verificado por igualdad
        exacta contra la constante _MENSAJE_*, no por subcadena laxa").
    (+) Distinguible por maquina (scripts/CI) sin parsear texto.
    (-) Dos exit codes mas que documentar; el humano en terminal no ve la
        distincion salvo que mire $LASTEXITCODE.

  Variante B — mismo exit (1) + prefijo "ERROR DE SINTAXIS YAML:" / "ERROR DE ESQUEMA:"
    (+) La distincion salta a la vista del humano sin mirar exit codes.
    (-) TENSION REAL: stderr ya NO es igual al mensaje del motor -> un test de
        igualdad exacta contra _MENSAJE_* falla; habria que aflojarlo a
        startswith/subcadena, que es justo lo que el contrato prohibe.

  Nota: A y B no son excluyentes (exit codes distintos + prefijo). Pero la
  combinacion hereda la (-) de B sobre la igualdad exacta.
""")

VARIANTE A — exit codes distintos, texto del motor INTACTO
cliente               excepcion                   exit  stderr (primera linea)
------------------------------------------------------------------------------
cliente_roto          ContractParseError          3     el archivo YAML del contrato es sintácticame
cliente_esquema       ContractSchemaError         4     archivo[1] 'ventas.csv': la lista de columna
cliente_scaffold      ContractSchemaError (M-01)  4     la clave raíz 'contract_data' debe contener 
cliente_ok            — (valido)                  0     (vacio)

VARIANTE B — mismo exit code (1), PREFIJO distinto
cliente               excepcion                   exit  stderr (primera linea)
------------------------------------------------------------------------------
cliente_roto          ContractParseError          1     ERROR DE SINTAXIS YAML: el archivo YAML del 
cliente_esquema       ContractSchemaError         1     ERROR DE ESQUEMA: archivo[1] 'ventas.csv': l
clie

## Celda 11 — HU-07: invocación mal formada → el **uso**, no un traceback

Cinco formas de escribir mal el comando. Todas deben imprimir `_USAGE` por **stderr** (con la línea nueva
del subcomando) y terminar sin excepción no controlada.

In [11]:
# =============================================================================
# Celda 11 — HU-07: invocaciones mal formadas -> el uso por stderr, sin reventar
# =============================================================================
MAL_FORMADAS = [
    (["contract"], "falta 'check' y el cliente"),
    (["contract", "check"], "falta el cliente"),
    (["contract", "foo", "cliente_ok"], "subcomando inexistente bajo 'contract'"),
    (["contract", "check", "cliente_ok", "de_mas"], "un argumento de mas"),
    ([], "sin argumentos"),
]
for argv, por_que in MAL_FORMADAS:
    code, out, err = mostrar(f"HU-07 — {por_que}", argv)
    assert code != 0, "una invocacion mal formada no puede terminar en exito"
    assert "uso: zlk" in err, "no se imprimio el uso por stderr"
    assert "contract check" in err, "_USAGE no anuncia el subcomando nuevo"

print("\nHU-07 OK: las 5 invocaciones mal formadas imprimen el uso por stderr y exit 1;")
print("ninguna levanto excepcion no controlada.")
print("\nOJO para la spec: 'cliente_ok de_mas' cae al uso SOLO porque el parser exige")
print("len(argv) == 3 EXACTO (igual que `_parse_client_new_name`). Con `>= 3` se")
print("tragaria argumentos de mas en silencio.")
print("\n_USAGE que propone el spike (agrega UNA linea al vigente):")
print(textwrap.indent(_USAGE_SPIKE, "    "))
print("\n_USAGE vigente hoy en cli.py:")
print(textwrap.indent(cli._USAGE, "    "))
print("\nRIESGO: si algun test vigente fija _USAGE por igualdad exacta, agregar la")
print("linea nueva lo rompe. Verificar en test_cli_client_new.py / test_cli_ingest.py")
print("antes de escribir el plan (la feature exige 'sin regresiones').")

$ zlk contract                                      [variante A]
  HU-07 — falta 'check' y el cliente
------------------------------------------------------------------------------
  exit code : 1
  stdout    : (vacio)
  stderr    : uso: zlk client new <NOMBRE_CLIENTE>
     zlk ingest <CLIENTE> <ruta>...
     zlk contract check <CLIENTE>
$ zlk contract check                                [variante A]
  HU-07 — falta el cliente
------------------------------------------------------------------------------
  exit code : 1
  stdout    : (vacio)
  stderr    : uso: zlk client new <NOMBRE_CLIENTE>
     zlk ingest <CLIENTE> <ruta>...
     zlk contract check <CLIENTE>
$ zlk contract foo cliente_ok                       [variante A]
  HU-07 — subcomando inexistente bajo 'contract'
------------------------------------------------------------------------------
  exit code : 1
  stdout    : (vacio)
  stderr    : uso: zlk client new <NOMBRE_CLIENTE>
     zlk ingest <CLIENTE> <ruta>...
     zlk con

## Celda 12 — HU-08: el comando **no escribe nada** en disco

Se toma una **foto** de todo el `clients_root` sintético (ruta → tamaño + `mtime_ns` + `sha256` de cada
archivo), se ejecutan **todas** las invocaciones del spike (válidas e inválidas, en ambas variantes) y se
vuelve a fotografiar. La igualdad de las dos fotos es la evidencia: el comando **diagnostica, no repara**
(D-22) — ni corrige el YAML roto, ni completa el vacío, ni toca `data/` ni `manifest.json`.

*(La foto lleva su propia guarda de no-vacuidad: comparar dos diccionarios vacíos no probaría nada.)*

In [12]:
# =============================================================================
# Celda 12 — HU-08: el comando es de SOLO LECTURA (no escribe nada en disco)
# =============================================================================
def foto(raiz):
    """Instantanea de TODO el arbol: ruta -> (bytes, mtime_ns, sha256)."""
    instantanea = {}
    for p in sorted(raiz.rglob("*")):
        if p.is_file():
            contenido = p.read_bytes()
            instantanea[str(p.relative_to(raiz))] = (
                len(contenido),
                p.stat().st_mtime_ns,
                hashlib.sha256(contenido).hexdigest(),
            )
    return instantanea

antes = foto(CLIENTS_ROOT)
# GUARDA DE NO-VACUIDAD: si la foto estuviera vacia, "nada cambio" seria vacuo.
assert antes, "foto vacia: la comparacion antes/despues no probaria nada"
print("archivos bajo el clients_root sintetico (guarda de no-vacuidad):", len(antes))

CASOS = ["cliente_ok", "cliente_scaffold", "cliente_esquema", "cliente_roto",
         "cliente_sin_contrato", "cliente_fantasma"]
invocaciones = 0
for cliente in CASOS:
    for variante in ("A", "B"):
        correr(["contract", "check", cliente], variante=variante)
        invocaciones += 1
print("invocaciones ejecutadas (validas e invalidas, ambas variantes):", invocaciones)

despues = foto(CLIENTS_ROOT)
nuevos = sorted(set(despues) - set(antes))
borrados = sorted(set(antes) - set(despues))
cambiados = [k for k in antes if k in despues and antes[k] != despues[k]]

print("\narchivos creados por el comando :", nuevos or "NINGUNO")
print("archivos borrados por el comando:", borrados or "NINGUNO")
print("archivos con contenido o mtime alterado:", cambiados or "NINGUNO")
assert antes == despues, "el comando escribio en disco: viola HU-08"
print("\nHU-08 OK: tras", invocaciones, "invocaciones, el arbol del tenant es identico")
print("   (mismo conjunto de archivos, mismo sha256 y mismo mtime_ns en cada uno)")
print("   en particular: el YAML invalido NO fue corregido y el vacio NO fue completado (D-22)")

archivos bajo el clients_root sintetico (guarda de no-vacuidad):

 25
invocaciones ejecutadas (validas e invalidas, ambas variantes): 12

archivos creados por el comando : NINGUNO
archivos borrados por el comando: NINGUNO
archivos con contenido o mtime alterado: NINGUNO

HU-08 OK: tras 12 invocaciones, el arbol del tenant es identico
   (mismo conjunto de archivos, mismo sha256 y mismo mtime_ns en cada uno)
   en particular: el YAML invalido NO fue corregido y el vacio NO fue completado (D-22)


## Celda 13 — HU-09: frontera D-21 · el comando **no lee ningún CSV** (con guarda de no-vacuidad, L-17)

Un `assert` negativo ("no tocó bronze") es **indistinguible de un instrumento roto** si nadie comprueba que
el instrumento observó algo. Por eso aquí cada espía trae su **guarda de no-vacuidad**: primero se exige que
el espía **haya visto** el YAML (prueba de que está vivo), y solo entonces vale la afirmación de ausencia.

**Técnica (L-17):** `builtins.open` **no** se parchea — en un kernel de Jupyter el espía queda mudo. Se usan
dos instrumentos independientes que **sí** funcionan:
1. **Audit hook de CPython** (`sys.addaudithook`, evento `open`): observa **toda** apertura del proceso, sin
   depender de cómo se resuelva el nombre `open`.
2. **Inyección en el namespace del módulo** (`mock.patch.object(contract_mod, "open", ..., create=True)`):
   crea un global `open` dentro de `contract.py`, que gana la búsqueda de nombres frente a builtins.

El tenant `cliente_ok` **tiene un CSV sintético real en `data/bronze/`**: si el comando lo abriera, se vería.

In [13]:
# =============================================================================
# Celda 13 — HU-09 / frontera D-21: el comando NO lee ningun CSV
# Dos espias INDEPENDIENTES, cada uno con su GUARDA DE NO-VACUIDAD (L-17).
# NO se parchea `builtins.open`: en un kernel de Jupyter el espia queda mudo
# y el assert negativo pasaria en vacio (L-17).
# =============================================================================
from unittest import mock
import builtins

# --- Instrumento 1: audit hook de CPython (evento "open") ---------------------
# Captura TODA apertura del proceso (venga del motor, de la fachada o de donde
# sea): no depende de en que namespace se resuelva el nombre `open`.
if "_ESPIA" not in globals():
    _ESPIA = {"activo": False, "aperturas": []}

    def _hook_open(event, args):
        if event == "open" and _ESPIA["activo"]:
            _ESPIA["aperturas"].append(str(args[0]))

    sys.addaudithook(_hook_open)
    print("audit hook instalado (una sola vez por kernel)")

_ESPIA["aperturas"].clear()
_ESPIA["activo"] = True
code_ok, _, _ = correr(["contract", "check", "cliente_ok"])
_ESPIA["activo"] = False
aperturas = list(_ESPIA["aperturas"])

print("\nInstrumento 1 — audit hook sobre el evento 'open'")
print("   el CSV sintetico SI existe en bronze:", CSV_BRONZE.exists(), "->", CSV_BRONZE.name)
print("   aperturas observadas durante `contract check cliente_ok`:", len(aperturas))
for a in aperturas:
    print("      -", a)

# GUARDA DE NO-VACUIDAD (L-17): un espia mudo haria pasar cualquier asercion de
# ausencia. Antes de afirmar "no leyo CSV", exigimos que el espia HAYA VISTO algo,
# y en concreto que haya visto el YAML del contrato.
assert aperturas, "ESPIA MUDO: no observo NINGUNA apertura -> la evidencia seria vacua"
vio_el_yaml = [a for a in aperturas if a.endswith("contract_data.yaml")]
assert vio_el_yaml, "ESPIA ROTO: ni siquiera vio el YAML que el motor abre con certeza"
print("   guarda de no-vacuidad: el espia observo", len(aperturas), "apertura(s) e incluye el YAML ->", vio_el_yaml)

# La asercion que importa, ya no vacua:
tocaron_datos = [a for a in aperturas if ("bronze" in a) or ("silver" in a)
                 or ("gold" in a) or a.endswith(".csv") or a.endswith("manifest.json")]
print("   aperturas bajo data/ o de CSV/manifest:", tocaron_datos or "NINGUNA")
assert not tocaron_datos, "el comando toco data/: se rompio la frontera D-21"
assert code_ok == 0

# --- Instrumento 2: espia inyectado en el namespace del MODULO del motor ------
# `mock.patch.object(contract_mod, "open", ...)` crea un global `open` en el
# modulo contract.py, que gana en la busqueda de nombres frente a builtins.
# Tecnica efectiva (a diferencia de parchear builtins) y acotada al motor.
_real_open = builtins.open
vistos = []

def _open_espia(archivo, *args, **kwargs):
    vistos.append(str(archivo))
    return _real_open(archivo, *args, **kwargs)

with mock.patch.object(contract_mod, "open", _open_espia, create=True):
    code2, out2, err2 = correr(["contract", "check", "cliente_ok"])

print("\nInstrumento 2 — espia inyectado en el namespace de zeroleak.config.contract")
print("   aperturas hechas POR EL MOTOR:", vistos)
assert vistos, "ESPIA MUDO: el motor no abrio nada -> instrumento inefectivo, evidencia vacua"
print("   guarda de no-vacuidad: el espia observo", len(vistos), "apertura(s) -> instrumento vivo")
print("   el motor abre EXACTAMENTE un archivo:", len(vistos) == 1)
print("   y ese archivo es el contrato del tenant:", vistos[0].endswith("contract_data.yaml"))
print("   el comando siguio funcionando con el espia puesto: exit", code2)
assert len(vistos) == 1 and vistos[0].endswith("contract_data.yaml")
assert code2 == 0

print("\nHU-09/D-21 OK: el resultado depende UNICAMENTE de contract_data.yaml.")

audit hook instalado (una sola vez por kernel)

Instrumento 1 — audit hook sobre el evento 'open'
   el CSV sintetico SI existe en bronze: True -> clientes.csv
   aperturas observadas durante `contract check cliente_ok`: 1
      - C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9\clients\cliente_ok\input\contract_data.yaml
   guarda de no-vacuidad: el espia observo 1 apertura(s) e incluye el YAML -> ['C:\\Users\\USUARIO\\AppData\\Local\\Temp\\zlk_spike_contract_check_xpie8cu9\\clients\\cliente_ok\\input\\contract_data.yaml']
   aperturas bajo data/ o de CSV/manifest: NINGUNA

Instrumento 2 — espia inyectado en el namespace de zeroleak.config.contract
   aperturas hechas POR EL MOTOR: ['C:\\Users\\USUARIO\\AppData\\Local\\Temp\\zlk_spike_contract_check_xpie8cu9\\clients\\cliente_ok\\input\\contract_data.yaml']
   guarda de no-vacuidad: el espia observo 1 apertura(s) -> instrumento vivo
   el motor abre EXACTAMENTE un archivo: True
   y ese archivo es el contrato del 

## Celda 14 — HU-10 (el motor intacto) · HU-11 (C-01) · resumen de trazas · limpieza

Cierre del spike. Verifica que **el motor no se tocó** (mismo `sha256` de `config/contract.py` al abrir y
al cerrar), que **todo el material fue sintético y temporal** (C-01), imprime el **mapa de trazas
`HU-xx` → celda → evidencia** que el humano usará en el gate, y **borra el temporal** (el spike no deja
residuos).

In [14]:
# =============================================================================
# Celda 14 — HU-10 (core intacto) · HU-11 (C-01) · resumen de trazas · limpieza
# =============================================================================
sha_final = hashlib.sha256(CONTRACT_PY.read_bytes()).hexdigest()
print("HU-10 — el motor no se toca")
print("   sha256(config/contract.py) al abrir el spike :", SHA_CONTRACT_INICIAL)
print("   sha256(config/contract.py) al cerrar el spike:", sha_final)
print("   intacto byte a byte durante todo el spike    :", sha_final == SHA_CONTRACT_INICIAL)
assert sha_final == SHA_CONTRACT_INICIAL
print("   (este notebook solo IMPORTA el motor; ninguna celda lo modifica)")

print("\nHU-11 (C-01) — Datos en Boveda: todo el material del spike es sintetico y temporal")
print("   directorio temporal    :", TMP)
print("   tenants sinteticos     :", sorted(p.name for p in CLIENTS_ROOT.iterdir()))
print("   filas de las 'matrices de mentiras' del CSV de bronze:")
for linea in CSV_BRONZE.read_text(encoding="utf-8").strip().splitlines():
    print("      ", linea)
assert all(str(p).startswith(str(TMP)) for p in TENANTS.values())
print("   todos los tenants viven bajo el temporal     :", True)
print("   ningun dato real de cliente entro al notebook: por construccion (nada se leyo de clients/)")

print("\n" + "=" * 78)
print("RESUMEN DE TRAZAS — cada HU de definition.md con evidencia visible arriba")
print("=" * 78)
_TRAZAS = [
    ("HU-01", "Celda 5",  "contrato valido multi-archivo -> exit 0 + confirmacion por stdout"),
    ("HU-02", "Celda 7",  "esquema invalido -> stderr == str(exc) del motor, verbatim"),
    ("HU-03", "Celda 8",  "YAML roto -> ContractParseError, distinguible del de esquema"),
    ("HU-04", "Celda 6",  "contrato recien scaffoldeado -> M-01 por igualdad EXACTA"),
    ("HU-05", "Celda 9",  "FileNotFoundError del motor demostrado + traducido por la fachada"),
    ("HU-06", "Celda 4",  "ZEROLEAK_CLIENTS_ROOT y fallback ./clients via cli._clients_root()"),
    ("HU-07", "Celda 11", "4 invocaciones mal formadas -> _USAGE por stderr, sin excepcion"),
    ("HU-08", "Celda 12", "12 invocaciones -> foto del arbol identica (sha256 + mtime)"),
    ("HU-09", "Celda 13", "2 espias con guarda de no-vacuidad -> solo se abre el YAML"),
    ("HU-10", "Celda 14", "sha256 de contract.py identico al abrir y al cerrar"),
    ("HU-11", "Celda 14", "tenants y YAMLs sinteticos en tempdir; nada real"),
    ("ABIERTO", "Celda 10", "parse vs schema: variante A (exit codes) vs B (prefijo) -> gate"),
]
print(f"{'HU':<9}{'donde':<11}evidencia")
print("-" * 78)
for hu, donde, que in _TRAZAS:
    print(f"{hu:<9}{donde:<11}{que}")

# --- Limpieza: el spike no deja residuos --------------------------------------
os.environ.pop("ZEROLEAK_CLIENTS_ROOT", None)
shutil.rmtree(TMP, ignore_errors=True)
print("\ntemporal eliminado:", not TMP.exists(), "|", TMP)
print("ZEROLEAK_CLIENTS_ROOT restaurada a su estado previo:",
      os.environ.get("ZEROLEAK_CLIENTS_ROOT") is None)

HU-10 — el motor no se toca
   sha256(config/contract.py) al abrir el spike : 564b44a3f046328cfcb8da977cd7c6b2de5caf0d8661f36e84f759f77754c118
   sha256(config/contract.py) al cerrar el spike: 564b44a3f046328cfcb8da977cd7c6b2de5caf0d8661f36e84f759f77754c118
   intacto byte a byte durante todo el spike    : True
   (este notebook solo IMPORTA el motor; ninguna celda lo modifica)

HU-11 (C-01) — Datos en Boveda: todo el material del spike es sintetico y temporal
   directorio temporal    : C:\Users\USUARIO\AppData\Local\Temp\zlk_spike_contract_check_xpie8cu9
   tenants sinteticos     : ['cliente_esquema', 'cliente_ok', 'cliente_roto', 'cliente_scaffold', 'cliente_sin_contrato']
   filas de las 'matrices de mentiras' del CSV de bronze:
       cliente_id,correo,alta
       1,test1@correo.com,2026-01-01
       2,test2@correo.com,2026-01-02
       3,test3@correo.com,2026-01-03
   todos los tenants viven bajo el temporal     : True
   ningun dato real de cliente entro al notebook: por constru

---

## Hallazgos del spike (insumo para `spec_writer`, paso 6)

1. **La fachada es realmente delgada.** El prototipo de `_dispatch_contract_check` cabe en ~15 líneas y no
   contiene ni un gramo de validación: resuelve la ruta con `cli._clients_root()`, comprueba dos
   precondiciones de existencia, invoca `load_contract` y traduce. Es el mismo patrón de `_dispatch_ingest`.

2. **El hueco del motor (HU-05) es real y está demostrado** (Celda 9): `load_contract` sobre un tenant
   inexistente propaga `FileNotFoundError` crudo. La traducción funciona **sin tocar el core** porque la
   fachada comprueba la precondición *antes* de invocarlo (D-22: el motor asume que el YAML existe).
   El spike distingue **dos** precondiciones (tenant ausente vs. contrato ausente) con **mensajes
   distintos y el mismo exit code 2**, imitando a `ingest`. *Pregunta para la spec:* ¿un solo exit code
   para ambas (como aquí), o dos?

3. **Punto abierto principal (Celda 10): cómo distinguir parse de schema.** Hay una tensión real entre
   el CA-2 del contrato ("mensaje del motor **verbatim**, verificado por igualdad exacta") y la variante
   B (prefijo): **con prefijo, `stderr` deja de ser igual al mensaje del motor por igualdad exacta**.
   La variante A (exit codes distintos, texto intacto) preserva las dos exigencias a la vez. El spike
   muestra ambas con salidas reales; **la decisión es del humano en el gate**.

4. **M-01 sale gratis (HU-04).** El contrato recién scaffoldeado (`contract_data:` sin cuerpo) ya produce
   M-01 exacto sin ningún código especial en la fachada: el fix de T-56 hace el trabajo. No necesita rama
   propia, solo un test que lo fije como camino de primera clase.

5. **La frontera D-21 se sostiene sola (HU-09).** El motor abre **exactamente un** archivo. Evidencia con
   dos instrumentos independientes y **guarda de no-vacuidad en ambos** (L-17). Técnica que **sí** funciona
   en kernel: *audit hook* de CPython (`sys.addaudithook`, evento `open`) e inyección del espía en el
   namespace del módulo (`mock.patch.object(contract_mod, "open", ..., create=True)`) — **no** se parchea
   `builtins.open`.

6. **Riesgo técnico detectado.** El `_USAGE` vigente es una constante congelada que los tests de CLI
   existentes podrían fijar por igualdad exacta: agregarle la línea de `contract check` puede romper
   tests vigentes. Debe verificarse en el plan (`test_cli_client_new.py` / `test_cli_ingest.py`) para no
   contradecir "sin regresiones" (CA-11 de la feature).

7. **Trampa de argumentos.** `zlk contract check <CLIENTE> extra` cae al uso solo porque el parser exige
   `len(argv) == 3` exacto. La spec debe fijarlo explícitamente (es fácil escribir `>= 3` e ingerir basura).

---

### Siguiente paso: **gate humano (paso 5)**
El humano revisa este spike y **aprueba** antes de pasar a `spec_writer` (paso 6). Este notebook es
**documentación de referencia**, no fuente de verdad: al construir `app/src/`, la spec y el bucle TDD
**reescriben** la lógica; no se copia-pega desde aquí.
